In [1]:
import random, json, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

random.seed(42)
NAMES = [("A", "B"), ("X", "Y"), ("P", "Q"), ("input 1", "input 2"), ("the first input", "the second input"),
         ("switch 1", "switch 2"), ("both inputs", None)]
OUT = ["output", "the output", "Y", "Z", "the signal", "the lamp"]
HI = ["high", "1", "HIGH", "on", "logic 1"]
LO = ["low", "0", "LOW", "off", "logic 0"]

T = {
 "AND": ["{o} is {h} only when both {a} and {b} are {h}",
         "{o} goes {h} if and only if {a} and {b} are both {h}",
         "{o} is {l} unless every input is {h}",
         "{o} is {h} when {a} and {b} are {h} together",
         "all inputs must be {h} for {o} to be {h}",
         "{o} stays {l} if any input is {l}",
         "{o} is {l} if either {a} or {b} is {l}, and {h} otherwise"],
 "OR":  ["{o} is {h} when at least one of {a} or {b} is {h}",
         "{o} is {h} if {a} or {b} or both are {h}",
         "{o} is {l} only when both {a} and {b} are {l}",
         "any {h} input makes {o} {h}",
         "{o} goes {l} only if every input is {l}",
         "{o} is {h} if either input is {h}"],
 "NAND":["{o} is {l} only when both {a} and {b} are {h}",
         "{o} is {h} unless {a} and {b} are both {h}",
         "{o} is the inverse of {a} AND {b}",
         "{o} goes {l} if and only if every input is {h}",
         "{o} is {h} if any input is {l}",
         "{o} is {h} when at least one of {a} or {b} is {l}"],
 "NOR": ["{o} is {h} only when both {a} and {b} are {l}",
         "{o} is {l} if either {a} or {b} is {h}",
         "{o} is the inverse of {a} OR {b}",
         "{o} goes {h} if and only if every input is {l}",
         "any {h} input forces {o} {l}",
         "{o} is {h} when neither {a} nor {b} is {h}"],
 "XOR": ["{o} is {h} when {a} and {b} are different",
         "{o} is {h} when exactly one of {a} or {b} is {h}",
         "{o} is {l} when {a} and {b} are equal",
         "{o} is {h} if the inputs differ and {l} if they match",
         "{o} is {h} when an odd number of inputs are {h}",
         "{o} is {h} if {a} or {b} is {h} but not both"],
 "XNOR":["{o} is {h} when {a} and {b} are the same",
         "{o} is {h} when both inputs are equal",
         "{o} is {l} when {a} and {b} are different",
         "{o} is {h} if the inputs match and {l} if they differ",
         "{o} is {h} when an even number of inputs are {h}",
         "{o} is the inverse of {a} XOR {b}"],
 "NOT": ["{o} is the opposite of {a}",
         "{o} is {h} when {a} is {l} and {l} when {a} is {h}",
         "{o} inverts {a}",
         "{o} is the complement of the single input {a}",
         "when {a} is {h}, {o} is {l}, and vice versa",
         "a single input is flipped to produce {o}"],
 "BUFFER":["{o} simply follows {a}",
         "{o} is {h} when {a} is {h} and {l} when {a} is {l}",
         "{o} copies the single input {a} without change",
         "{o} is identical to {a}; no inversion occurs",
         "the single input {a} is passed straight through to {o}",
         "{o} equals {a}"],
}
GATES = list(T)

def fill(t):
    a, b = random.choice(NAMES)
    if b is None:
        a, b = "input 1", "input 2"
    return t.format(a=a, b=b, o=random.choice(OUT), h=random.choice(HI), l=random.choice(LO))

def build(split_seed=0):
    Xtr, ytr, Xte, yte = [], [], [], []
    for g, ts in T.items():
        ts = ts[:]
        random.Random(split_seed).shuffle(ts)
        train_t, test_t = ts[:-2], ts[-2:]
        for t in train_t:
            for _ in range(40): Xtr.append(fill(t)); ytr.append(g)
        for t in test_t:
            for _ in range(20): Xte.append(fill(t)); yte.append(g)
    return Xtr, ytr, Xte, yte

def norm(s):
    s = s.lower()
    s = re.sub(r"\b(logic 1|high|on|1)\b", " HI ", s)
    return re.sub(r"\b(logic 0|low|off|0)\b", " LO ", s)

def seen_template_split():
    X, y = [], []
    for g, ts in T.items():
        for t in ts:
            for _ in range(30): X.append(fill(t)); y.append(g)
    idx = list(range(len(X))); random.Random(1).shuffle(idx); c = int(.8 * len(idx))
    pick = lambda ids: ([X[i] for i in ids], [y[i] for i in ids])
    return pick(idx[:c]) + pick(idx[c:])

def make_model():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 3), preprocessor=norm),
                         LogisticRegression(max_iter=2000, C=10))

Xs, ys, Xt, yt = seen_template_split()
seen_acc = accuracy_score(yt, make_model().fit(Xs, ys).predict(Xt))
print(f"[A] Seen-template split accuracy:   {seen_acc:.3f}")
accs = []
for sd in range(5):
    a, b, c_, d = build(sd)
    accs.append(accuracy_score(d, make_model().fit(a, b).predict(c_)))
print(f"[B] Unseen-template accuracy (5 splits): {[round(x,3) for x in accs]}  mean={sum(accs)/5:.3f}\n")

Xtr, ytr, Xte, yte = build()
model = make_model().fit(Xtr, ytr)
pred = model.predict(Xte)
print(f"Train size: {len(Xtr)}  Test size: {len(Xte)}")
print(classification_report(yte, pred, digits=3, zero_division=0))

demo = ["Output is high only when both inputs are high",
        "The lamp lights if at least one switch is pressed",
        "Output is low only if both inputs are high",
        "Output is high when the two inputs are not equal",
        "Output flips whatever the single input is",
        "Output is high only when both inputs are low"]
print("Demo:")
for s in demo:
    print(f"  {s!r} -> {model.predict([s])[0]} ({max(model.predict_proba([s])[0]):.2f})")

[A] Seen-template split accuracy:   1.000
[B] Unseen-template accuracy (5 splits): [0.053, 0.0, 0.006, 0.003, 0.072]  mean=0.027

Train size: 1320  Test size: 320
              precision    recall  f1-score   support

         AND      0.000     0.000     0.000        40
      BUFFER      0.298     0.425     0.351        40
        NAND      0.000     0.000     0.000        40
         NOR      0.000     0.000     0.000        40
         NOT      0.000     0.000     0.000        40
          OR      0.000     0.000     0.000        40
        XNOR      0.000     0.000     0.000        40
         XOR      0.000     0.000     0.000        40

    accuracy                          0.053       320
   macro avg      0.037     0.053     0.044       320
weighted avg      0.037     0.053     0.044       320

Demo:
  'Output is high only when both inputs are high' -> AND (0.59)
  'The lamp lights if at least one switch is pressed' -> OR (0.94)
  'Output is low only if both inputs are high' ->

In [2]:
sentence = "Output is high only when both inputs are high"

prediction = model.predict([sentence])[0]
sentence
print("Input:", sentence)
print("Predicted Output:", prediction)

Input: Output is high only when both inputs are high
Predicted Output: AND


In [3]:
sentence = "Output is high only when both inputs are low"

prediction = model.predict([sentence])[0]

print("Input:", sentence)
print("Predicted Output:", prediction)

Input: Output is high only when both inputs are low
Predicted Output: NOR


In [4]:
sentence = "The lamp lights if at least one switch is pressed"

prediction = model.predict([sentence])[0]
sentence
print("Input:", sentence)
print("Predicted Output:", prediction)

Input: The lamp lights if at least one switch is pressed
Predicted Output: OR


In [5]:
sentence = "Output is low only if both inputs are high"

prediction = model.predict([sentence])[0]
sentence
print("Input:", sentence)
print("Predicted Output:", prediction)

Input: Output is low only if both inputs are high
Predicted Output: XNOR
